# 06 Q3: โหลดกระจุกที่ไหน

Pain point ของ BART: Transbay Tube คือคอขวดด้านความจุของทั้งระบบ ต้องรู้ว่าทิศไหนและชั่วโมงไหนที่กำหนดจำนวนตู้ที่ต้องใช้

ข้อมูลที่ใช้
- fact table (`od`) ระดับ OD-ชั่วโมง สำหรับ flow ข้ามอ่าวรายชั่วโมง (query ตรงจาก Parquet 112 ล้านแถว)
- od_month และ station_hour สำหรับคู่ OD และสถานี
- วันทำงาน normal เทียบ ม.ค.-เม.ย. 2019 กับ ม.ค.-เม.ย. 2023

ทริปข้ามอ่าว = สถานีต้นทางกับปลายทางอยู่คนละฝั่งอ่าว (ทุกทริปแบบนี้ต้องผ่าน Transbay Tube)

ข้อจำกัด
- feed มีชั่วโมงเดียวต่อทริป จึงนับทริปที่ชั่วโมงที่เข้าสถานี ทริปข้ามอ่าวส่วนใหญ่ใช้เวลาน้อยกว่า 1 ชั่วโมง โหลดใน Tube จริงจึงอาจเหลื่อมไปชั่วโมงถัดไปบ้าง
- ไม่มีข้อมูลจำนวนขบวนและความจุ จึงวัดได้แค่ความต้องการ

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from bart import viz
from bart.config import create_views, duck, fetch_value
from bart.perf import stage
from bart.periods import AFTER, BEFORE, PERIODS, period_case

viz.notebook_mode()

con = duck()
create_views(con)

## 1. ทริปข้ามอ่าวรายชั่วโมง แยกทิศ

query จาก fact table ตรงๆ เพราะ marts ไม่มี OD รายชั่วโมง
ไฟล์ Parquet เรียงตามวันที่ไว้แล้ว DuckDB จึงอ่านแค่ row group ของช่วง ม.ค.-เม.ย. ไม่ต้องอ่านทั้ง 112 ล้านแถว

ค่าที่ได้คือทริปเฉลี่ยต่อวันทำงานในแต่ละชั่วโมง

In [2]:
date_filter_parts = []
for start, end in PERIODS.values():
    date_filter_parts.append(f"o.date BETWEEN DATE '{start}' AND DATE '{end}'")
date_filter = " OR ".join(date_filter_parts)

crossing_query = f"""
    CREATE OR REPLACE TEMP TABLE crossing_hour AS
    WITH workday_trips AS (
        SELECT
            {period_case("o.date")}           AS period,
            o.hour,
            so.side || ' to ' || sd.side        AS direction,
            o.trips
        FROM od AS o
        JOIN calendar AS c ON c.date = o.date
        JOIN stations AS so ON so.station = o.origin
        JOIN stations AS sd ON sd.station = o.dest
        WHERE ({date_filter})
          AND c.day_status = 'normal'
          AND c.is_workday
          AND o.date >= so.opening_date
          AND o.date >= sd.opening_date
    ),
    workdays AS (
        SELECT {period_case("date")} AS period, count(*) AS days
        FROM calendar
        WHERE day_status = 'normal' AND is_workday
        GROUP BY period
    )
    SELECT
        t.period,
        t.direction,
        t.hour,
        sum(t.trips) / w.days AS trips_per_day
    FROM workday_trips AS t
    JOIN workdays AS w ON w.period = t.period
    GROUP BY t.period, t.direction, t.hour, w.days
"""

with stage("06.crossing_hour_from_fact"):
    con.execute(crossing_query)

con.sql("""
    SELECT period, direction, round(sum(trips_per_day)) AS trips_per_day
    FROM crossing_hour
    GROUP BY period, direction
    ORDER BY period, direction
""").df()

[perf] 06.crossing_hour_from_fact: 0.21s | 10.36 cores | peak RAM 0.283 GB | CPU 55.8%


,period,direction,trips_per_day
0,Jan-Apr 2019,east to east,79512.0
1,Jan-Apr 2019,east to west,113886.0
2,Jan-Apr 2019,west to east,113978.0
3,Jan-Apr 2019,west to west,102479.0
4,Jan-Apr 2023,east to east,36502.0
5,Jan-Apr 2023,east to west,39657.0
6,Jan-Apr 2023,west to east,38946.0
7,Jan-Apr 2023,west to west,36792.0


In [3]:
crossing = con.sql("""
    SELECT period, direction, hour, trips_per_day
    FROM crossing_hour
    WHERE direction IN ('east to west', 'west to east')
    ORDER BY period, direction, hour
""").df()

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=["East Bay to SF / Peninsula", "SF / Peninsula to East Bay"])

period_styles = {BEFORE: dict(color=viz.CONTEXT, width=3), AFTER: dict(color=viz.ACCENT, width=3)}
column = 1
for direction in ["east to west", "west to east"]:
    for period, style in period_styles.items():
        rows = crossing[(crossing["period"] == period) & (crossing["direction"] == direction)]
        fig.add_trace(
            go.Scatter(
                x=rows["hour"],
                y=rows["trips_per_day"],
                mode="lines+markers",
                name=period,
                legendgroup=period,
                showlegend=(column == 1),
                line=style,
                marker=dict(size=5),
                hovertemplate="%{x}:00 - %{y:,.0f} trips<extra>" + period + "</extra>",
            ),
            row=1,
            col=column,
        )
    column += 1

fig.update_xaxes(title_text="hour", dtick=3)
fig.update_yaxes(title_text="trips crossing the Bay per workday hour", tickformat="~s", row=1, col=1)
fig.update_layout(hovermode="closest", margin=dict(t=170), legend=dict(y=1.16))

peak_2019 = fetch_value(con, f"SELECT max(trips_per_day) FROM crossing_hour WHERE period = '{BEFORE}'")
peak_2023 = fetch_value(con, f"SELECT max(trips_per_day) FROM crossing_hour WHERE period = '{AFTER}'")
title = f"The busiest Tube hour fell from {peak_2019 / 1000:.1f}k to {peak_2023 / 1000:.1f}k trips ({100 * peak_2023 / peak_2019:.0f}% of 2019)"
subtitle = "Trips crossing the Bay by entry hour and direction, average normal workday, Jan-Apr 2019 vs Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "06_crossing_by_hour", height=560)
fig

## 2. ชั่วโมง-ทิศที่กำหนดความจุ

จำนวนตู้ที่ต้องใช้ขึ้นกับชั่วโมงและทิศที่คนเยอะที่สุด (peak direction) ส่วนทิศตรงข้ามในชั่วโมงเดียวกัน รถจะวิ่งค่อนข้างว่าง

ดู 5 อันดับแรกของ ชั่วโมง × ทิศ และอัตราส่วนระหว่างทิศหลักกับทิศสวน

In [4]:
top_slots = con.sql(f"""
    WITH ranked AS (
        SELECT
            period,
            direction,
            hour,
            trips_per_day,
            row_number() OVER (PARTITION BY period ORDER BY trips_per_day DESC) AS slot_rank
        FROM crossing_hour
        WHERE direction IN ('east to west', 'west to east')
    )
    SELECT period, slot_rank, direction, hour, round(trips_per_day) AS trips_per_day
    FROM ranked
    WHERE slot_rank <= 5
    ORDER BY period, slot_rank
""").df()

top_slots

,period,slot_rank,direction,hour,trips_per_day
0,Jan-Apr 2019,1,east to west,8,25137.0
1,Jan-Apr 2019,2,west to east,18,23051.0
2,Jan-Apr 2019,3,west to east,17,22257.0
3,Jan-Apr 2019,4,east to west,9,20300.0
4,Jan-Apr 2019,5,east to west,7,16530.0
5,Jan-Apr 2023,1,east to west,8,7948.0
6,Jan-Apr 2023,2,west to east,17,7537.0
7,Jan-Apr 2023,3,west to east,18,6255.0
8,Jan-Apr 2023,4,east to west,7,5596.0
9,Jan-Apr 2023,5,east to west,9,5550.0


In [5]:
imbalance = con.sql(f"""
    WITH pivoted AS (
        SELECT
            period,
            hour,
            sum(trips_per_day) FILTER (WHERE direction = 'east to west') AS east_to_west,
            sum(trips_per_day) FILTER (WHERE direction = 'west to east') AS west_to_east
        FROM crossing_hour
        GROUP BY period, hour
    )
    SELECT
        period,
        CASE WHEN hour BETWEEN 6 AND 9 THEN 'AM peak 6-9' ELSE 'PM peak 15-18' END AS peak,
        round(sum(east_to_west)) AS east_to_west,
        round(sum(west_to_east)) AS west_to_east,
        round(greatest(sum(east_to_west), sum(west_to_east))
              / least(sum(east_to_west), sum(west_to_east)), 1) AS main_to_counter_ratio
    FROM pivoted
    WHERE hour BETWEEN 6 AND 9 OR hour BETWEEN 15 AND 18
    GROUP BY period, peak
    ORDER BY period, peak
""").df()

imbalance

,period,peak,east_to_west,west_to_east,main_to_counter_ratio
0,Jan-Apr 2019,AM peak 6-9,70319.0,8658.0,8.1
1,Jan-Apr 2019,PM peak 15-18,13771.0,65372.0,4.7
2,Jan-Apr 2023,AM peak 6-9,22150.0,3206.0,6.9
3,Jan-Apr 2023,PM peak 15-18,5884.0,21943.0,3.7


## 3. คู่ OD ที่หนาแน่นที่สุด

top 20 คู่ OD ของวันทำงานปี 2019 และผู้โดยสารปี 2023 ของคู่เดียวกัน (จาก od_month หารด้วยจำนวนวันใน month_days)

In [6]:
od_period_query = f"""
    CREATE OR REPLACE TEMP TABLE od_period AS
    WITH workday_months AS (
        SELECT
            {period_case("o.month")} AS period,
            o.origin::VARCHAR          AS origin,
            o.dest::VARCHAR            AS dest,
            o.trips,
            d.days
        FROM od_month AS o
        JOIN month_days AS d ON d.month = o.month AND d.day_type = o.day_type
        WHERE o.day_type = 'workday'
    )
    SELECT period, origin, dest, sum(trips) / sum(days) AS trips_per_day
    FROM workday_months
    WHERE period IS NOT NULL
    GROUP BY period, origin, dest
"""

con.execute(od_period_query)

top_pairs = con.sql(f"""
    SELECT
        b.origin,
        b.dest,
        so.side <> sd.side                          AS crosses_bay,
        round(b.trips_per_day)                      AS trips_2019,
        round(a.trips_per_day)                      AS trips_2023,
        round(100 * a.trips_per_day / b.trips_per_day, 1) AS recovery_pct
    FROM od_period AS b
    JOIN od_period AS a ON a.origin = b.origin AND a.dest = b.dest AND a.period = '{AFTER}'
    JOIN stations AS so ON so.station = b.origin
    JOIN stations AS sd ON sd.station = b.dest
    WHERE b.period = '{BEFORE}'
    ORDER BY b.trips_per_day DESC
    LIMIT 20
""").df()

top_pairs

,origin,dest,crosses_bay,trips_2019,trips_2023,recovery_pct
0,BALB,MONT,False,2653.0,691.0,26.0
1,24TH,MONT,False,2472.0,711.0,28.7
2,DUBL,EMBR,True,2459.0,642.0,26.1
3,WOAK,EMBR,True,2253.0,792.0,35.2
4,POWL,BALB,False,2241.0,846.0,37.7
5,MONT,24TH,False,2107.0,606.0,28.8
6,PHIL,EMBR,True,2092.0,498.0,23.8
7,DALY,MONT,False,2071.0,644.0,31.1
8,EMBR,DUBL,True,2038.0,577.0,28.3
9,24TH,EMBR,False,1946.0,670.0,34.4


In [7]:
ordered = list(reversed(top_pairs.to_dict("records")))

labels = []
trips_2019 = []
trips_2023 = []
recovery = []
segment_x = []
segment_y = []
for row in ordered:
    label = f"{row['origin']} to {row['dest']}"
    labels.append(label)
    trips_2019.append(row["trips_2019"])
    trips_2023.append(row["trips_2023"])
    recovery.append(row["recovery_pct"])
    segment_x.extend([row["trips_2023"], row["trips_2019"], None])
    segment_y.extend([label, label, None])

segments = go.Scatter(x=segment_x, y=segment_y, mode="lines", line=dict(color=viz.GRID, width=4),
                      showlegend=False, hoverinfo="skip")
points_2019 = go.Scatter(
    x=trips_2019, y=labels, mode="markers", name=BEFORE,
    marker=dict(color=viz.CONTEXT, size=11),
    hovertemplate="%{y}: %{x:,.0f} trips per workday<extra>2019</extra>",
)
points_2023 = go.Scatter(
    x=trips_2023, y=labels, mode="markers", name=AFTER,
    marker=dict(color=viz.ACCENT, size=11),
    customdata=recovery,
    hovertemplate="%{y}: %{x:,.0f} trips per workday (%{customdata:.0f}% of 2019)<extra>2023</extra>",
)

fig = go.Figure([segments, points_2019, points_2023])
fig.update_layout(hovermode="closest", margin=dict(l=140))
fig.update_xaxes(title_text="trips per workday", tickformat="~s", showgrid=True, gridcolor=viz.GRID, rangemode="tozero")
fig.update_yaxes(showgrid=False)

crossing_count = 0
for row in ordered:
    if row["crosses_bay"]:
        crossing_count += 1
title = f"The 20 busiest OD pairs of 2019 kept only {top_pairs['trips_2023'].sum() / top_pairs['trips_2019'].sum() * 100:.0f}% of their riders"
subtitle = f"Average trips per normal workday, Jan-Apr 2019 vs Jan-Apr 2023. {crossing_count} of the 20 pairs cross the Bay"
viz.titled(fig, title, subtitle)
viz.save(fig, "06_top_od_pairs", height=700)
fig

## 4. สถานีที่รับและส่งคนตอนเช้า

net flow ช่วงเช้า (6:00-9:59) = exits - entries ต่อวันทำงาน
- บวก = คนออกจากสถานีมากกว่าเข้า (ปลายทางตอนเช้า)
- ลบ = คนเข้าสถานีมากกว่าออก (ต้นทางตอนเช้า)

ขนาดของ net flow บอกว่าแต่ละสถานีดึงคนเข้าหรือปล่อยคนออกตอนเช้าเท่าไร ซึ่งกำหนดว่าขบวนจะแน่นในช่วงไหนของเส้นทาง

In [8]:
net_flow_query = f"""
    WITH morning AS (
        SELECT
            h.station::VARCHAR        AS station,
            {period_case("h.date")} AS period,
            h.date,
            sum(h.exits) - sum(h.entries) AS net_flow
        FROM station_hour AS h
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal' AND c.is_workday AND h.hour BETWEEN 6 AND 9
        GROUP BY h.station, h.date
    ),
    averages AS (
        SELECT station, period, avg(net_flow) AS net_flow
        FROM morning
        WHERE period IS NOT NULL
        GROUP BY station, period
    )
    SELECT
        s.station,
        s.name,
        s.side,
        round(b.net_flow) AS net_flow_2019,
        round(a.net_flow) AS net_flow_2023
    FROM stations AS s
    LEFT JOIN averages AS b ON b.station = s.station AND b.period = '{BEFORE}'
    LEFT JOIN averages AS a ON a.station = s.station AND a.period = '{AFTER}'
    ORDER BY a.net_flow DESC
"""

net_flow = con.sql(net_flow_query).df()

receiving = net_flow[net_flow["net_flow_2023"] > 0]
print("morning destinations in 2023:", ", ".join(receiving["station"]))
net_flow.head(12)

morning destinations in 2023: EMBR, MONT, CIVC, POWL, DBRK, 12TH, SFIA, 16TH, 19TH, OAKL, WARM


,station,name,side,net_flow_2019,net_flow_2023
0,EMBR,Embarcadero,west,30770.0,8686.0
1,MONT,Montgomery St,west,30060.0,7489.0
2,CIVC,Civic Center/UN Plaza,west,10557.0,3245.0
3,POWL,Powell St,west,9489.0,3188.0
4,DBRK,Downtown Berkeley,east,1855.0,1061.0
5,12TH,12th St Oakland City Center,east,4812.0,797.0
6,SFIA,SFO Airport,west,1235.0,657.0
7,16TH,16th St Mission,west,1318.0,590.0
8,19TH,19th St Oakland,east,2618.0,193.0
9,OAKL,Oakland Airport,east,57.0,37.0


In [9]:
def flow_size(row):
    return abs(row["net_flow_2023"])


def flow_value(row):
    return row["net_flow_2023"]


flow_rows = net_flow.to_dict("records")
flow_rows.sort(key=flow_size, reverse=True)
chosen = flow_rows[:16]
chosen.sort(key=flow_value)

labels = []
values_2019 = []
values_2023 = []
for row in chosen:
    labels.append(f"{row['station']}  {row['name']}")
    values_2019.append(row["net_flow_2019"])
    values_2023.append(row["net_flow_2023"])

bars_2019 = go.Bar(name=BEFORE, y=labels, x=values_2019, orientation="h", marker_color=viz.CONTEXT,
                   hovertemplate="%{y}: %{x:+,.0f} per morning<extra>2019</extra>")
bars_2023 = go.Bar(name=AFTER, y=labels, x=values_2023, orientation="h", marker_color=viz.ACCENT,
                   hovertemplate="%{y}: %{x:+,.0f} per morning<extra>2023</extra>")

fig = go.Figure([bars_2019, bars_2023])
fig.update_layout(barmode="group", hovermode="closest", margin=dict(l=260))
fig.add_vline(x=0, line=dict(color=viz.MUTED, width=1))
fig.update_xaxes(title_text="exits minus entries, 6:00-9:59, per workday", tickformat="~s", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)

downtown_2019 = 0.0
downtown_2023 = 0.0
for row in net_flow.to_dict("records"):
    if row["station"] in ["EMBR", "MONT", "POWL", "CIVC"]:
        downtown_2019 += row["net_flow_2019"]
        downtown_2023 += row["net_flow_2023"]

title = f"Downtown SF still takes in {100 * downtown_2023 / downtown_2019:.0f}% of its 2019 morning inflow"
subtitle = "Net morning flow (exits - entries, 6:00-9:59) per workday, 16 stations with the largest flow in 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "06_morning_net_flow", height=720)
fig

## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้คือ Transbay Tube เป็นคอขวดของทั้งระบบ ต้องรู้ว่าชั่วโมงและทิศไหนที่กำหนดจำนวนตู้ที่ต้องใช้

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | ชั่วโมง-ทิศที่หนาแน่นที่สุดของ Tube ยังเป็นช่วงเดิม คือ East Bay ไป SF ตอน 8:00 แต่เหลือ 7,948 ทริปต่อวันทำงาน (ปี 2019 คือ 25,137 หรือเหลือ 32%) | ชั่วโมง-ทิศที่กำหนดความจุ | ตรงจุด | ใช้ 8:00 ทิศไปทางตะวันตกเป็นชั่วโมงหลักในการกำหนดขนาดขบวน ความต้องการสูงสุดตอนนี้อยู่ราว 1 ใน 3 ของที่ระบบเคยรองรับ |
| 2 | ช่วงเย็นยอดสูงสุดเลื่อนจาก 18:00 มาเป็น 17:00 (ทิศไปทางตะวันออก 7,537 ทริป) ส่วน 18:00 เหลือ 27% ของปี 2019 | ชั่วโมง-ทิศที่กำหนดความจุ | ตรงจุด | ย้ายขบวนเสริมช่วงเย็นมาไว้ที่ 17:00 และลดที่ 18:00-19:00 สอดคล้องกับ Q2 |
| 3 | ทิศสวนยังว่างมาก ช่วง peak เช้า ทิศไปทางตะวันตกมากกว่าทิศกลับ 6.9 เท่า (ปี 2019 คือ 8.1 เท่า) ช่วงเย็น 3.7 เท่า (เดิม 4.7) | ขบวนเปล่าตอนกลับทิศ | ตรงจุด | ขบวนที่วิ่งสวนทิศใน peak ใช้ขบวนสั้นลงได้ ช่องว่างนี้แคบลงเล็กน้อยแต่ยังใหญ่ |
| 4 | 20 คู่ OD ที่หนาแน่นที่สุดของปี 2019 เหลือ 30% และครึ่งหนึ่งข้ามอ่าว คู่ที่ฟื้นน้อยที่สุดคือการเดินทางไกลจากชานเมืองเข้า downtown (DUBL และ PHIL กับ MONT/EMBR เหลือ 20-26%) ส่วนคู่ที่สั้นกว่า เช่น WOAK กับ EMBR เหลือ 35-38% | โหลดกระจุกที่ไหน | ตรงจุด | ขบวนยาวจากปลายสาย Dublin และ Concord ช่วง peak เป็นจุดแรกที่ลดขนาดได้ |
| 5 | downtown SF (EMBR, MONT, CIVC, POWL) ยังเป็นปลายทางหลักตอนเช้า แต่รับคนได้ 28% ของปี 2019 downtown Oakland แทบหายไปจากการเป็นปลายทาง (12TH เหลือ 17%, 19TH เหลือ 7%) | โหลดกระจุกที่ไหน | ตรงจุด | สถานี downtown Oakland ช่วงเช้าไม่จำเป็นต้องรองรับการลงจำนวนมากเหมือนเดิม |
| 6 | WARM เปลี่ยนจากต้นทางตอนเช้า (-1,448) มาเป็นปลายทางเล็กน้อย (+31) | การเปลี่ยนบทบาทของสถานี | บางส่วน สอดคล้องกับ Q1 และ EDA แต่สาเหตุต้องดูใน Q4 | ตรวจใน Q4 |

ข้อจำกัด
- นับทริปที่ชั่วโมงที่เข้าสถานี โหลดใน Tube จริงอาจเหลื่อมไปชั่วโมงถัดไป
- ไม่มีข้อมูลขบวนและความจุ ตัวเลขทุกตัวคือความต้องการ ไม่ใช่ความแออัด

ต่อไป: 07_q4_new_stations